# Data cleaning: MELD and EmoryNLP

This notebook documents and checks every cleaning step applied to the two datasets of the project.
The cleaning itself runs in `prepare_data.py` (code in `data/loaders.py`), and the duplicate/overlap
checks come from `data/audit.py`. The notebook re-derives each step from the raw files, shows what it
changes, and verifies that `data/processed/` matches. It does not write or modify any data.

**Decisions**

- The official train/dev/test splits are kept. No utterance is removed or relabelled.
- Text keeps its punctuation, casing and negation; only leading/trailing whitespace is stripped.
- Overlaps and duplicates that come from the datasets themselves are *flagged* in
  `outputs/flagged_dialogues.csv` for sensitivity analyses, not removed.

Run `python prepare_data.py` first, then run all cells with the project's `.venv` kernel.

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
while not (ROOT / "prepare_data.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("open this notebook from inside the repository")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from data.audit import find_duplicate_dialogues, find_scene_overlaps, flagged_dialogues, normalize_text
from data.loaders import (DATASETS, EXPECTED_COUNTS, LABELS, RAW_FILES, RECORD_FIELDS, SPLITS,
                          load_raw_split, load_split, parse_meld_speakers, raw_path)
from prepare_data import SOURCES, sha256

RAW, PROCESSED = ROOT / "data" / "raw", ROOT / "data" / "processed"
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 200)

## 1. Inventory: two datasets, six files

The project uses exactly two corpora, MELD and the original text release of EmoryNLP, each with its
three official splits. (`data/pretrained/` holds the DistilBERT weights, which are a model, not data.)

In [2]:
rows = []
for ds in DATASETS:
    for sp in SPLITS:
        path = raw_path(RAW, ds, sp)
        rows.append({"dataset": ds, "split": sp, "file": path.name, "MB": round(path.stat().st_size / 1e6, 2),
                     "matches pinned official file (SHA-256)": sha256(path) == SOURCES[ds]["sha256"][path.name]})
inventory = pd.DataFrame(rows)

found = sorted(p.relative_to(RAW).as_posix() for p in RAW.rglob("*") if p.is_file())
expected = sorted(f"{ds}/{RAW_FILES[ds][sp]}" for ds in DATASETS for sp in SPLITS)
assert found == expected, found
assert inventory.iloc[:, -1].all()
print(f"{len(DATASETS)} datasets ({', '.join(DATASETS)}), {len(found)} raw files, nothing else in data/raw/")
inventory

2 datasets (meld, emorynlp), 6 raw files, nothing else in data/raw/


,dataset,split,file,MB,matches pinned official file (SHA-256)
0,meld,train,train_sent_emo.csv,1.12,True
1,meld,dev,dev_sent_emo.csv,0.12,True
2,meld,test,test_sent_emo.csv,0.29,True
3,emorynlp,train,emotion-detection-trn.json,4.23,True
4,emorynlp,dev,emotion-detection-dev.json,0.56,True
5,emorynlp,test,emotion-detection-tst.json,0.56,True


## 2. MELD

One CSV row per utterance: `Utterance, Speaker, Emotion, Sentiment, Dialogue_ID, Utterance_ID, Season,
Episode, StartTime, EndTime`. Only the text, speaker, emotion and the two ids are used; `Sentiment` and
the video metadata are not model inputs.

In [3]:
meld_raw = {sp: pd.read_csv(raw_path(RAW, "meld", sp), dtype=str, keep_default_na=False, encoding="utf-8")
            for sp in SPLITS}
display(pd.DataFrame({sp: {"utterances": len(df), "dialogues": df.Dialogue_ID.nunique(),
                           "proposal": EXPECTED_COUNTS["meld"][sp]} for sp, df in meld_raw.items()}).T)
meld_raw["train"].head()

,utterances,dialogues,proposal
train,9989,1038,9989
dev,1109,114,1109
test,2610,280,2610


,Sr No.,Utterance,Speaker,Emotion,Sentiment,Dialogue_ID,Utterance_ID,Season,Episode,StartTime,EndTime
0,1,also I was the point person on my company’s transition from the KL-5 to GR-6 system.,Chandler,neutral,neutral,0,0,8,21,"00:16:16,059","00:16:21,731"
1,2,You must’ve had your hands full.,The Interviewer,neutral,neutral,0,1,8,21,"00:16:21,940","00:16:23,442"
2,3,That I did. That I did.,Chandler,neutral,neutral,0,2,8,21,"00:16:23,442","00:16:26,389"
3,4,So let’s talk a little bit about your duties.,The Interviewer,neutral,neutral,0,3,8,21,"00:16:26,820","00:16:29,572"
4,5,My duties? All right.,Chandler,surprise,positive,0,4,8,21,"00:16:34,452","00:16:40,917"


### 2.1 Missing values and "NA"-like text

With default settings pandas turns strings such as `NA`, `None` or `null` into missing values. The loader
reads every column as text with `keep_default_na=False`, so an utterance like "None." could never be lost.

In [4]:
na_like = {"", "na", "n/a", "nan", "none", "null"}
pd.DataFrame({sp: {
    "missing cells with pandas defaults": int(pd.read_csv(raw_path(RAW, "meld", sp)).isna().sum().sum()),
    "empty or NA-like utterances": int(df.Utterance.str.strip().str.lower().isin(na_like).sum()),
} for sp, df in meld_raw.items()}).T

,missing cells with pandas defaults,empty or NA-like utterances
train,0,0
dev,0,0
test,0,0


### 2.2 Whitespace and encoding

In [5]:
def text_checks(texts: pd.Series) -> dict:
    return {
        "leading/trailing whitespace (stripped)": int((texts != texts.str.strip()).sum()),
        "repeated inner spaces (kept)": int(texts.str.contains("  ", regex=False).sum()),
        "utterances with non-ASCII characters (kept)": int(texts.map(lambda t: any(ord(c) > 127 for c in t)).sum()),
        "mojibake such as \u00e2\u20ac or \ufffd": int(texts.str.contains("\u00e2\u20ac|\u00c3|\ufffd").sum()),
    }

display(pd.DataFrame({sp: text_checks(df.Utterance) for sp, df in meld_raw.items()}).T)
Counter(c for df in meld_raw.values() for t in df.Utterance for c in t if ord(c) > 127).most_common(10)

,leading/trailing whitespace (stripped),repeated inner spaces (kept),utterances with non-ASCII characters (kept),mojibake such as â€ or �
train,0,637,2740,0
dev,0,62,308,0
test,0,165,757,0


[('’', 4966),
 ('…', 198),
 ('—', 189),
 ('‘', 115),
 ('“', 8),
 ('”', 6),
 ('é', 6),
 ('–', 2),
 ('ó', 1)]

The files are clean UTF-8 (the GitHub CSVs, unlike some older MELD copies, have no mojibake). Curly quotes,
ellipses and dashes are kept in the stored text; the DistilBERT tokenizer splits them like their ASCII
forms, and the TF-IDF baseline maps curly apostrophes to `'` inside its vectoriser.

### 2.3 Speakers

Joint turns are written as "Joey and Chandler". They become a list of both speakers, so the
same-speaker relation can use overlap. Group names such as "All" and "Both" stay literal speaker ids.

In [6]:
spk = pd.concat(df.assign(split=sp) for sp, df in meld_raw.items())
joint = spk[spk.Speaker.str.contains(" and ", regex=False)]
display(pd.DataFrame({"split": joint.split, "raw Speaker": joint.Speaker,
                      "speaker_ids": joint.Speaker.map(parse_meld_speakers)}).reset_index(drop=True))
spk[spk.Speaker.isin(["All", "Both"])].groupby(["Speaker", "split"]).size().unstack(fill_value=0)

,split,raw Speaker,speaker_ids
0,train,Joey and Chandler,"[Joey, Chandler]"
1,train,Monica and Ross,"[Monica, Ross]"
2,train,Phoebe and Rachel,"[Phoebe, Rachel]"
3,train,Rachel and Phoebe,"[Rachel, Phoebe]"
4,train,Ross and Joey,"[Ross, Joey]"
5,train,Monica and Phoebe,"[Monica, Phoebe]"
6,train,Joey and Ross,"[Joey, Ross]"
7,dev,Monica and Rachel,"[Monica, Rachel]"
8,dev,Ross and Rachel,"[Ross, Rachel]"
9,dev,Ross and Rachel,"[Ross, Rachel]"


split,dev,test,train
Speaker,,,
All,10,8,33
Both,0,0,1


### 2.4 Labels

In [7]:
meld_labels = pd.concat({sp: df.Emotion.value_counts() for sp, df in meld_raw.items()}, axis=1)
assert set(meld_labels.index) == set(LABELS["meld"])
meld_labels.loc[LABELS["meld"]]

,train,dev,test
Emotion,,,
anger,1109,153,345
disgust,271,22,68
fear,268,40,50
joy,1743,163,402
neutral,4710,470,1256
sadness,683,111,208
surprise,1205,150,281


### 2.5 Order and utterance-id gaps

Some dialogues skip utterance ids (utterances removed by the MELD authors). Ids are kept as released;
rows are ordered by `(Dialogue_ID, Utterance_ID)` and `turn_index` records the position, so
"the previous k turns" is well defined.

In [8]:
def gap_dialogues(df: pd.DataFrame) -> pd.Series:
    ids = df.assign(u=df.Utterance_ID.astype(int)).groupby("Dialogue_ID").u.apply(sorted)
    return ids[ids.map(lambda u: u != list(range(len(u))))]

for sp, df in meld_raw.items():
    keys = list(zip(df.Dialogue_ID.astype(int), df.Utterance_ID.astype(int)))
    print(f"{sp:5s}: file already in (Dialogue_ID, Utterance_ID) order: {keys == sorted(keys)}; "
          f"dialogues with id gaps: {len(gap_dialogues(df))}")
gap_dialogues(meld_raw["train"]).head(3).to_frame("Utterance_IDs present")

train: file already in (Dialogue_ID, Utterance_ID) order: True; dialogues with id gaps: 72
dev  : file already in (Dialogue_ID, Utterance_ID) order: True; dialogues with id gaps: 2
test : file already in (Dialogue_ID, Utterance_ID) order: True; dialogues with id gaps: 2


,Utterance_IDs present
Dialogue_ID,
1,"[0, 1, 2, 3, 4, 5, 8]"
1022,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 15, 16, 17]"
107,"[0, 1, 2, 4, 5, 6, 7, 8, 9, 10]"


### 2.6 Metadata quirks (not model inputs)

`StartTime` mixes `0:10:04,312` and `00:10:04,312`. More importantly, the episode metadata is not
always right: test dialogue 84 is labelled season 1, episode 5, but its lines and timestamps are the
season 6, episode 4 scene that train dialogue 12 also contains. This is why section 5 matches scenes by
text, not by episode.

In [9]:
for sp, df in meld_raw.items():
    print(f"{sp:5s}: StartTime without a leading zero: {int((~df.StartTime.str.match(r'^\d\d:')).sum())}")
cols = ["Dialogue_ID", "Utterance_ID", "Speaker", "Utterance", "Emotion", "Season", "Episode", "StartTime"]
pd.concat({
    "test": meld_raw["test"].query("Dialogue_ID == '84'")[cols],
    "train": meld_raw["train"].query("Dialogue_ID == '12' and Utterance_ID in ['4', '5', '6']")[cols],
})

train: StartTime without a leading zero: 2412
dev  : StartTime without a leading zero: 270
test : StartTime without a leading zero: 685


Dialogue_ID Utterance_ID   Speaker                                                                                            Utterance  Emotion Season Episode     StartTime
test  745          84            0  Chandler                                                                               What’s wrong with you?    anger      1       5  00:10:04,312
      746          84            1      Joey                                                                                             Nothing!    anger      1       5  00:10:07,023
      747          84            2      Joey  Well, I-I got this blinding pain in my stomach when I was lifting weights before, then I uh pass...  sadness      1       5  00:10:08,190
train 127          12            4  Chandler                                                                               What’s wrong with you?  disgust      6       4   0:10:05,647
      128          12            5      Joey                                                                                             Nothing!  neutral      6       4  00:10:07,023
      129          12            6      Joey  Well, I-I got this blinding pain in my stomach when I was lifting weights before, then I uh pass...     fear      6       4  00:10:08,190

## 3. EmoryNLP (original text release)

JSON: season → episodes → scenes → utterances with `utterance_id, speakers, transcript, tokens, emotion`.
One scene is one dialogue. Only the transcript, speakers, emotion and ids are used.

In [10]:
emory_raw = {}
for sp in SPLITS:
    with open(raw_path(RAW, "emorynlp", sp), encoding="utf-8") as f:
        season = json.load(f)
    emory_raw[sp] = pd.DataFrame([
        {"episode_id": ep["episode_id"], "scene_id": sc["scene_id"], **u}
        for ep in season["episodes"] for sc in ep["scenes"] for u in sc["utterances"]
    ])
display(pd.DataFrame({sp: {"utterances": len(df), "scenes": df.scene_id.nunique(), "episodes": df.episode_id.nunique(),
                           "proposal": EXPECTED_COUNTS["emorynlp"][sp]} for sp, df in emory_raw.items()}).T)
emory_raw["train"][["scene_id", "utterance_id", "speakers", "transcript", "emotion"]].head()

,utterances,scenes,episodes,proposal
train,9934,713,77,9934
dev,1344,99,11,1344
test,1328,85,9,1328


,scene_id,utterance_id,speakers,transcript,emotion
0,s01_e02_c01,s01_e02_c01_u001,[Monica Geller],"What you guys don't understand is, for us, kissing is as important as any part of it.",Joyful
1,s01_e02_c01,s01_e02_c01_u002,[Joey Tribbiani],"Yeah, right!.......Y'serious?",Neutral
2,s01_e02_c01,s01_e02_c01_u003,[Phoebe Buffay],"Oh, yeah!",Joyful
3,s01_e02_c01,s01_e02_c01_u004,[Rachel Green],Everything you need to know is in that first kiss.,Powerful
4,s01_e02_c01,s01_e02_c01_u005,[Monica Geller],Absolutely.,Powerful


### 3.1 Text, speakers and labels

In [11]:
display(pd.DataFrame({sp: text_checks(df.transcript) for sp, df in emory_raw.items()}).T)
display(pd.DataFrame({sp: {
    "multi-speaker utterances (all kept)": int((df.speakers.str.len() > 1).sum()),
    "no speaker": int((df.speakers.str.len() == 0).sum()),
    "same name twice in one list (de-duplicated)": int(df.speakers.map(lambda s: len(s) != len(set(s))).sum()),
} for sp, df in emory_raw.items()}).T)
emory_labels = pd.concat({sp: df.emotion.value_counts() for sp, df in emory_raw.items()}, axis=1)
assert set(emory_labels.index.str.lower()) == set(LABELS["emorynlp"])
emory_labels  # capitalised in the release; lower-cased by the loader

,leading/trailing whitespace (stripped),repeated inner spaces (kept),utterances with non-ASCII characters (kept),mojibake such as â€ or �
train,0,0,0,0
dev,0,0,0,0
test,0,0,0,0


,multi-speaker utterances (all kept),no speaker,same name twice in one list (de-duplicated)
train,56,0,0
dev,5,0,0
test,10,0,0


,train,dev,test
emotion,,,
Neutral,3034,393,349
Joyful,2184,289,282
Scared,1285,178,182
Mad,1076,143,113
Peaceful,900,132,159
Powerful,784,134,145
Sad,671,75,98


### 3.2 Order and episodes

In [12]:
for sp, df in emory_raw.items():
    in_order = df.groupby("scene_id", sort=False).utterance_id.apply(lambda s: list(s) == sorted(s)).all()
    print(f"{sp:5s}: utterance ids increase inside every scene: {in_order}")
eps = {sp: set(df.episode_id) for sp, df in emory_raw.items()}
print("episodes shared train/dev, train/test, dev/test:",
      len(eps["train"] & eps["dev"]), len(eps["train"] & eps["test"]), len(eps["dev"] & eps["test"]))

train: utterance ids increase inside every scene: True
dev  : utterance ids increase inside every scene: True
test : utterance ids increase inside every scene: True
episodes shared train/dev, train/test, dev/test: 0 0 0


EmoryNLP's splits are episode-disjoint (MELD's are not), but section 5 shows that a dev episode still
re-uses train scenes.

## 4. Cleaning applied, and verification of `data/processed/`

Each step below is counted against the raw files. No row is dropped in any split.

In [13]:
processed = {ds: {sp: load_split(PROCESSED, ds, sp) for sp in SPLITS} for ds in DATASETS}

def moved(raw_ids, recs):
    return sum(a != r["utterance_id"] for a, r in zip(raw_ids, recs))

steps = []
for sp in SPLITS:
    m, e = meld_raw[sp], emory_raw[sp]
    steps.append({"dataset": "meld", "split": sp, "rows in": len(m), "rows out": len(processed["meld"][sp]),
                  "whitespace stripped": int((m.Utterance != m.Utterance.str.strip()).sum()),
                  "speaker lists split / de-duplicated": int(m.Speaker.str.contains(" and ", regex=False).sum()),
                  "label lower-cased": int((m.Emotion != m.Emotion.str.lower()).sum()),
                  "rows moved by ordering": moved("dia" + m.Dialogue_ID + "_utt" + m.Utterance_ID, processed["meld"][sp])})
    steps.append({"dataset": "emorynlp", "split": sp, "rows in": len(e), "rows out": len(processed["emorynlp"][sp]),
                  "whitespace stripped": int((e.transcript != e.transcript.str.strip()).sum()),
                  "speaker lists split / de-duplicated": int(e.speakers.map(lambda s: len(s) != len(set(s))).sum()),
                  "label lower-cased": int((e.emotion != e.emotion.str.lower()).sum()),
                  "rows moved by ordering": moved(e.utterance_id, processed["emorynlp"][sp])})
pd.DataFrame(steps).set_index(["dataset", "split"])

,,rows in,rows out,whitespace stripped,speaker lists split / de-duplicated,label lower-cased,rows moved by ordering
dataset,split,,,,,,
meld,train,9989,9989,0,7,0,0
emorynlp,train,9934,9934,0,0,9934,0
meld,dev,1109,1109,0,4,0,0
emorynlp,dev,1344,1344,0,0,1344,0
meld,test,2610,2610,0,3,0,0
emorynlp,test,1328,1328,0,0,1328,0


In [14]:
for ds in DATASETS:
    for sp in SPLITS:
        recs = processed[ds][sp]
        assert recs == load_raw_split(RAW, ds, sp), f"{ds} {sp}: data/processed is stale, rerun prepare_data.py"
        assert all(tuple(r) == RECORD_FIELDS for r in recs)
        assert len({r["utterance_id"] for r in recs}) == len(recs)
        assert all(r["label"] in LABELS[ds] and r["text"] for r in recs)
        turns = {}
        for r in recs:
            turns.setdefault(r["dialogue_id"], []).append(r["turn_index"])
        assert all(t == list(range(len(t))) for t in turns.values())
print("data/processed matches the cleaning code for all 6 splits: schema, unique ids, valid labels, "
      "non-empty text, turn_index 0..n-1 in order")
pd.DataFrame(processed["meld"]["train"][:3])

data/processed matches the cleaning code for all 6 splits: schema, unique ids, valid labels, non-empty text, turn_index 0..n-1 in order


,dataset,split,dialogue_id,utterance_id,turn_index,speaker_ids,text,label
0,meld,train,dia0,dia0_utt0,0,[Chandler],also I was the point person on my company’s transition from the KL-5 to GR-6 system.,neutral
1,meld,train,dia0,dia0_utt1,1,[The Interviewer],You must’ve had your hands full.,neutral
2,meld,train,dia0,dia0_utt2,2,[Chandler],That I did. That I did.,neutral


## 5. Duplicates and overlaps (kept, flagged)

### 5.1 Same text, different gold labels

Short interjections carry different labels in different contexts. This is inherent ambiguity rather
than an error, and it is exactly what the context models try to resolve, so these rows are kept.

In [15]:
for ds in DATASETS:
    tr = pd.DataFrame(processed[ds]["train"]).assign(norm=lambda d: d.text.map(normalize_text))
    labels = tr.groupby("norm").label.agg(lambda s: dict(Counter(s).most_common()))
    conflicts = labels[labels.map(len) > 1]
    print(f"{ds} train: {len(conflicts)} texts have more than one gold label ({tr.norm.isin(conflicts.index).sum()} rows)")
    display(conflicts.to_frame("labels").assign(rows=conflicts.map(lambda c: sum(c.values())))
            .sort_values("rows", ascending=False).head(5))

meld train: 105 texts have more than one gold label (877 rows)


,labels,rows
norm,,
hey!,"{'joy': 45, 'neutral': 19, 'surprise': 11, 'anger': 4}",79
okay.,"{'neutral': 63, 'joy': 1}",64
what?,"{'surprise': 39, 'neutral': 21, 'sadness': 2, 'fear': 1}",63
yeah.,"{'neutral': 45, 'sadness': 1, 'joy': 1}",47
no.,"{'neutral': 36, 'sadness': 1}",37


emorynlp train: 164 texts have more than one gold label (1125 rows)


,labels,rows
norm,,
what?,"{'neutral': 47, 'scared': 32, 'mad': 8, 'joyful': 6, 'powerful': 3, 'sad': 3}",99
hey.,"{'neutral': 56, 'joyful': 8, 'scared': 2, 'peaceful': 2, 'sad': 1, 'mad': 1}",70
hey!,"{'joyful': 36, 'neutral': 13, 'peaceful': 6, 'powerful': 2, 'scared': 1}",58
yeah.,"{'neutral': 43, 'joyful': 4, 'peaceful': 4, 'sad': 1, 'powerful': 1}",53
hi.,"{'neutral': 39, 'peaceful': 5, 'joyful': 3, 'mad': 1, 'scared': 1}",49


### 5.2 Duplicate dialogues inside one split

In [16]:
dups = find_duplicate_dialogues(processed)
display(dups)
for r in dups.itertuples():
    split = pd.DataFrame(processed[r.dataset][r.split])
    display(split[split.dialogue_id.isin([r.duplicate_of, r.dialogue_id])]
            [["dialogue_id", "turn_index", "speaker_ids", "text", "label"]])

,dataset,split,dialogue_id,duplicate_of,n_utterances
0,emorynlp,dev,s04_e01_c14,s04_e01_c13,5


,dialogue_id,turn_index,speaker_ids,text,label
914,s04_e01_c13,0,[Monica Geller],Gin.,neutral
915,s04_e01_c13,1,[Chandler Bing],"We were playing Gin? Y'know if we were a couple, we could play this game naked.",joyful
916,s04_e01_c13,2,[Monica Geller],Will you stop!,mad
917,s04_e01_c13,3,[Chandler Bing],Okay. All right.,powerful
918,s04_e01_c13,4,[Monica Geller],"Okay, all right, I think you're great, I think you're sweet, and you're smart, and I love you. B...",powerful
919,s04_e01_c14,0,[Monica Geller],Gin.,joyful
920,s04_e01_c14,1,[Chandler Bing],"We were playing Gin? Y'know if we were a couple, we could play this game naked.",joyful
921,s04_e01_c14,2,[Monica Geller],Will you stop!,scared
922,s04_e01_c14,3,[Chandler Bing],Okay. All right.,neutral
923,s04_e01_c14,4,[Monica Geller],"Okay, all right, I think you're great, I think you're sweet, and you're smart, and I love you. B...",peaceful


### 5.3 Scenes re-used across splits

A dev/test utterance matches an earlier-split dialogue when both contain a rare 6-word sequence or the
same rare text of at least 4 words ("rare" = found in at most 3 dialogues of the dataset; spelled-out
words like "C.H.E.E.S.E." are ignored). A dialogue counts as a re-used scene when at least 2 distinct
texts match, or more than half of its utterances with 5+ words match. Matching on word sequences
survives re-segmentation and wrong episode metadata, while common phrases ("oh my god, oh my god")
do not trigger it.

In [17]:
overlaps = find_scene_overlaps(processed)
by_scene = (overlaps.assign(same_label=overlaps.label == overlaps.matched_label)
            .groupby(["dataset", "split", "dialogue_id", "matched_split", "matched_dialogue_id"])
            .agg(matched_utterances=("utterance_id", "size"), same_gold_label=("same_label", "sum")))
display(by_scene)
overlaps[["dataset", "split", "utterance_id", "label", "matched_utterance_id", "matched_label", "text"]]

matched_utterances  same_gold_label
dataset  split dialogue_id matched_split matched_dialogue_id                                     
emorynlp dev   s04_e21_c06 train         s01_e07_c11                           5                2
               s04_e21_c07 train         s01_e05_c15                           5                4
               s04_e21_c08 train         s02_e07_c11                           6                5
               s04_e21_c09 train         s03_e15_c07                          10                0
                                         s03_e16_c01                           6                1
meld     test  dia84       train         dia12                                 2                0
               dia85       train         dia12                                 4                2

,dataset,split,utterance_id,label,matched_utterance_id,matched_label,text
0,meld,test,dia84_utt0,anger,dia12_utt4,disgust,What’s wrong with you?
1,meld,test,dia84_utt2,sadness,dia12_utt6,fear,"Well, I-I got this blinding pain in my stomach when I was lifting weights before, then I uh pass..."
2,meld,test,dia85_utt0,neutral,dia12_utt7,neutral,"But um, I don’t think it’s anything serious."
3,meld,test,dia85_utt1,surprise,dia12_utt8,surprise,This sounds like a hernia. You have to—you-you—Go to the doctor!
4,meld,test,dia85_utt3,anger,dia12_utt9,fear,"‘Kay look, if I have to go to the doctor for anything it’s gonna be for this thing sticking out ..."
5,meld,test,dia85_utt4,disgust,dia12_utt11,anger,Why did I have to start working out again?
6,emorynlp,dev,s04_e21_c06_u002,neutral,s01_e07_c11_u002,joyful,"Hey, I have a question. Well, actually, it's not so much a question as.. more of a general wonde..."
7,emorynlp,dev,s04_e21_c06_u004,neutral,s01_e07_c11_u004,neutral,"Okay. Umm, for a while now, I've been wanting to, um...."
8,emorynlp,dev,s04_e21_c06_u006,neutral,s01_e07_c11_u006,neutral,"Yes, yes, that's right..."
9,emorynlp,dev,s04_e21_c06_u007,joyful,s01_e07_c11_u007,peaceful,"Oh, look at the little cat! Look at it!"


- **MELD test `dia84`, `dia85`** are the same scene as **train `dia12`** (season 6, episode 4), cut into
  different dialogues and partly re-segmented. Most matched lines got a different gold label in train.
- **EmoryNLP dev `s04_e21_c06`–`c09`** come from a clip-show episode that replays scenes of training
  episodes (s01e05, s01e07, s02e07, s03e15/s03e16). The test split has no re-used scene.

### 5.4 Shared single lines that are not flagged

Identical long lines can also appear in unrelated scenes, e.g. a recorded phone message that plays
several times in one episode. They are listed for transparency but are not scene overlaps.

In [18]:
flagged_keys = set(zip(overlaps.dataset, overlaps.split, overlaps.dialogue_id))
rows = []
for ds in DATASETS:
    train_text = {}
    for r in processed[ds]["train"]:
        train_text.setdefault(normalize_text(r["text"]), r["utterance_id"])
    for sp in ("dev", "test"):
        for r in processed[ds][sp]:
            t = normalize_text(r["text"])
            if len(t.split()) >= 8 and t in train_text and (ds, sp, r["dialogue_id"]) not in flagged_keys:
                rows.append({"dataset": ds, "split": sp, "utterance_id": r["utterance_id"],
                             "train_utterance_id": train_text[t], "text": r["text"]})
pd.DataFrame(rows)

,dataset,split,utterance_id,train_utterance_id,text
0,meld,test,dia225_utt0,dia776_utt4,"Thank you for your patience, you’re the next caller."


### 5.5 Flagged list for sensitivity analyses

`prepare_data.py` writes this table to `outputs/flagged_dialogues.csv`. Main results use the full official
splits; a sensitivity check recomputes the dev/test metrics without these dialogues (join on
`dataset, split, dialogue_id`).

In [19]:
flagged = flagged_dialogues(processed)
saved = pd.read_csv(ROOT / "outputs" / "flagged_dialogues.csv")
assert saved.to_csv(index=False) == flagged.to_csv(index=False), "outputs/flagged_dialogues.csv is stale"
display(flagged)
impact = []
for (ds, sp), g in flagged.groupby(["dataset", "split"]):
    n_utt = sum(1 for r in processed[ds][sp] if r["dialogue_id"] in set(g.dialogue_id))
    impact.append({"dataset": ds, "split": sp, "flagged dialogues": g.dialogue_id.nunique(),
                   "utterances": n_utt, "share of split": f"{100 * n_utt / len(processed[ds][sp]):.2f}%"})
pd.DataFrame(impact)

,dataset,split,dialogue_id,n_utterances,issue,matched_split,matched_dialogue_id,n_matched_utterances
0,emorynlp,dev,s04_e01_c14,5,duplicate_within_split,dev,s04_e01_c13,5
1,emorynlp,dev,s04_e21_c06,8,same_scene_as_train,train,s01_e07_c11,5
2,emorynlp,dev,s04_e21_c07,5,same_scene_as_train,train,s01_e05_c15,5
3,emorynlp,dev,s04_e21_c08,10,same_scene_as_train,train,s02_e07_c11,6
4,emorynlp,dev,s04_e21_c09,11,same_scene_as_train,train,s03_e15_c07,10
5,emorynlp,dev,s04_e21_c09,11,same_scene_as_train,train,s03_e16_c01,6
6,meld,test,dia84,3,same_scene_as_train,train,dia12,2
7,meld,test,dia85,6,same_scene_as_train,train,dia12,4


,dataset,split,flagged dialogues,utterances,share of split
0,emorynlp,dev,5,39,2.90%
1,meld,test,2,9,0.34%


## 6. Final cleaned data

In [20]:
display(pd.DataFrame([{"dataset": ds, "split": sp, "utterances": len(recs),
                       "dialogues": len({r["dialogue_id"] for r in recs}),
                       "matches proposal": len(recs) == EXPECTED_COUNTS[ds][sp]}
                      for ds in DATASETS for sp, recs in processed[ds].items()]))
for ds in DATASETS:
    dist = pd.DataFrame({sp: pd.Series(Counter(r["label"] for r in recs)) for sp, recs in processed[ds].items()})
    display((100 * dist / dist.sum()).round(1).loc[LABELS[ds]].rename_axis(f"{ds} label (% of split)"))

,dataset,split,utterances,dialogues,matches proposal
0,meld,train,9989,1038,True
1,meld,dev,1109,114,True
2,meld,test,2610,280,True
3,emorynlp,train,9934,713,True
4,emorynlp,dev,1344,99,True
5,emorynlp,test,1328,85,True


,train,dev,test
meld label (% of split),,,
anger,11.1,13.8,13.2
disgust,2.7,2.0,2.6
fear,2.7,3.6,1.9
joy,17.4,14.7,15.4
neutral,47.2,42.4,48.1
sadness,6.8,10.0,8.0
surprise,12.1,13.5,10.8


,train,dev,test
emorynlp label (% of split),,,
joyful,22.0,21.5,21.2
mad,10.8,10.6,8.5
neutral,30.5,29.2,26.3
peaceful,9.1,9.8,12.0
powerful,7.9,10.0,10.9
sad,6.8,5.6,7.4
scared,12.9,13.2,13.7


**Outputs of the cleaning step** (all written by `python prepare_data.py`):

| file | content |
|---|---|
| `data/processed/<dataset>/<split>.jsonl` | cleaned records: `dataset, split, dialogue_id, utterance_id, turn_index, speaker_ids, text, label` |
| `data/processed/manifest.json` | source URLs, checksums, counts |
| `outputs/data_audit.md` | full audit report |
| `outputs/flagged_dialogues.csv` | dialogues to leave out in sensitivity analyses |